# NOISe detector baseline for boneseg

Retrains NOISe's osteoclast model (YOLOv8-L segmentation, as in github.com/michaelwwan/noise) once per held-out batch,
so that it never sees the batch it is scored on, and predicts the same 20 test patches per batch that boneseg was
scored on. Training data: 1500 random patches of each of the other four batches at 416 px (NOISe trains at 416),
validation on 100 more of each; a fixed budget of 1.3 GPU-hours per fold. Output: `preds.zip` with one 16-bit
instance map per test patch, scored locally by `paper/noise_yolo_score.py`.

In [ ]:
!pip install -q ultralytics stream-unzip
import io, json, os, random, shutil, time, zipfile
from pathlib import Path
import numpy as np, requests, torch
from PIL import Image
from stream_unzip import stream_unzip
URL = "https://drive.usercontent.google.com/download?id=1WgIKqd346BtTpU_N5lBzIFw0zPHOXNOP&export=download&confirm=t"
TEST = json.loads('''{"m1": ["img_5_0_8", "img_5_16_5", "img_15_2_6", "img_3_12_12", "img_21_12_1", "img_1_5_11", "img_21_5_18", "img_17_15_13", "img_11_5_9", "img_9_1_4", "img_3_11_16", "img_11_12_14", "img_13_11_9", "img_7_17_4", "img_1_12_0", "img_7_15_13", "img_9_7_13", "img_9_14_0", "img_5_6_12", "img_5_13_12"], "m2": ["img_17_14_13", "img_21_10_0", "img_11_7_8", "img_13_11_7", "img_3_19_7", "img_5_18_12", "img_13_3_8", "img_7_8_5", "img_13_7_8", "img_9_19_12", "img_11_18_4", "img_7_19_7", "img_13_0_8", "img_9_12_2", "img_3_5_6", "img_9_19_10", "img_7_15_12", "img_21_7_18", "img_3_2_11", "img_19_8_2"], "m3": ["img_13_10_12", "img_9_5_7", "img_11_3_9", "img_1_4_18", "img_1_13_9", "img_17_1_10", "img_3_10_4", "img_21_11_14", "img_13_5_6", "img_11_4_18", "img_5_9_10", "img_33_18_11", "img_15_7_4", "img_23_4_7", "img_13_9_9", "img_21_17_4", "img_15_10_13", "img_25_7_13", "img_29_14_1", "img_33_6_3"], "m4": ["img_19_6_14", "img_23_8_3", "img_9_8_2", "img_3_18_20", "img_9_16_9", "img_3_15_20", "img_13_6_8", "img_5_20_6", "img_19_12_16", "img_1_12_7", "img_7_11_17", "img_1_18_4", "img_9_11_9", "img_3_5_9", "img_13_11_14", "img_11_10_18", "img_23_9_20", "img_15_21_13", "img_5_16_15", "img_19_17_14"], "m5": ["img_7_8_10", "img_1_2_9", "img_3_13_14", "img_1_5_17", "img_7_7_2", "img_9_11_2", "img_3_14_11", "img_3_8_13", "img_3_5_7", "img_9_15_5", "img_3_12_16", "img_1_3_14", "img_7_10_12", "img_5_19_10", "img_7_9_8", "img_1_19_3", "img_1_12_0", "img_3_5_13", "img_5_8_10", "img_1_5_11"]}''')
ROOT, TESTDIR, OUT = Path("/tmp/noise"), Path("/tmp/test"), Path("/kaggle/working")
BATCHES = ["m1", "m2", "m3", "m4", "m5"]
test_names = {(b, n) for b in TEST for n in TEST[b]}
print(torch.cuda.device_count(), "GPUs")

In [ ]:
# Stream the 18 GB zip once: osteoclast labels (class 0) of every patch, images resized to 416 px,
# and the original 832 px test patches
t0 = time.time()
for b in BATCHES:
    (ROOT / b / "images").mkdir(parents=True, exist_ok=True); (ROOT / b / "labels").mkdir(parents=True, exist_ok=True)
    (TESTDIR / b).mkdir(parents=True, exist_ok=True)
def drive_stream(file_id):
    """Streams a large Google Drive file. From cloud machines Drive often answers with an HTML page (a virus-scan
    warning or confirmation form) instead of the file; its form fields are then sent back to get the file itself."""
    import requests, re as _re
    s = requests.Session()
    url = "https://drive.usercontent.google.com/download"
    params = {"id": file_id, "export": "download", "confirm": "t"}
    for attempt in range(4):
        r = s.get(url, params=params, stream=True, timeout=120)
        if "text/html" not in r.headers.get("Content-Type", ""):
            return r
        page = r.text
        fields = dict(_re.findall(r'name="([^"]+)" value="([^"]*)"', page))
        action = _re.search(r'action="([^"]+)"', page)
        print("Drive answered with a page; resubmitting its form", sorted(fields), flush=True)
        if action: url = action.group(1).replace("&amp;", "&")
        params = {**params, **fields}
    raise RuntimeError("Google Drive did not return the file: " + page[:300])
def chunks():
    with drive_stream("1WgIKqd346BtTpU_N5lBzIFw0zPHOXNOP") as r:
        for c in r.iter_content(1 << 20):
            yield c
n = 0
for name, _, data in stream_unzip(chunks()):
    name = name.decode()
    parts = name.split("/")
    if len(parts) == 4 and parts[0] == "Mouse" and parts[2] in ("images", "labels") and parts[3] and not parts[3].startswith("."):
        b, kind, stem = parts[1], parts[2], Path(parts[3]).stem
        raw = b"".join(data)
        if kind == "labels":
            lines = [l for l in raw.decode().splitlines() if l.startswith("0 ")]
            (ROOT / b / "labels" / f"{stem}.txt").write_text("\n".join(lines))
        else:
            im = Image.open(io.BytesIO(raw)).convert("RGB")
            if (b, stem) in test_names:
                im.save(TESTDIR / b / f"{stem}.png")
            im.resize((416, 416), Image.BILINEAR).save(ROOT / b / "images" / f"{stem}.jpg", quality=95)
            n += 1
            if n % 5000 == 0: print(n, "images", round(time.time() - t0), "s", flush=True)
    else:
        for _ in data: pass
print("done", n, "images in", round(time.time() - t0), "s")

In [ ]:
# One fold per held-out batch
from ultralytics import YOLO
device = 0
preds = OUT / "preds"; preds.mkdir(exist_ok=True)
for held in BATCHES:
    rng = random.Random(0)
    train, val = [], []
    for b in BATCHES:
        if b == held: continue
        imgs = sorted(str(p) for p in (ROOT / b / "images").glob("*.jpg"))
        rng.shuffle(imgs)
        train += imgs[:1500]; val += imgs[1500:1600]
    lists = Path(f"/tmp/fold_{held}"); lists.mkdir(exist_ok=True)
    (lists / "train.txt").write_text("\n".join(train)); (lists / "val.txt").write_text("\n".join(val))
    (lists / "data.yaml").write_text(f"path: {lists}\ntrain: {lists}/train.txt\nval: {lists}/val.txt\nnames:\n  0: Osteoclast\n")
    t0 = time.time()
    model = YOLO("yolov8l-seg.pt")
    model.train(data=str(lists / "data.yaml"), imgsz=416, epochs=50, time=1.3, batch=16, device=device, project=str(OUT / "runs"),
                name=f"fold_{held}", exist_ok=True, plots=False, workers=4, verbose=False)
    print(held, "trained in", round((time.time() - t0) / 60), "min", flush=True)
    best_path = Path(model.trainer.best)   # Where the trainer saved it (ultralytics may nest run folders)
    best = YOLO(str(best_path))
    (preds / held).mkdir(exist_ok=True)
    for f in sorted((TESTDIR / held).glob("*.png")):
        r = best(str(f), imgsz=416, retina_masks=True, verbose=False, device=0)[0]
        h, w = r.orig_shape
        inst = np.zeros((h, w), np.uint16)
        if r.masks is not None:
            m = r.masks.data.cpu().numpy() > 0.5
            order = np.argsort(r.boxes.conf.cpu().numpy())   # Most confident drawn last, so it wins overlaps
            for k, i in enumerate(order, start=1):
                inst[m[i][:h, :w]] = k
        Image.fromarray(inst).save(preds / held / f.name)
    shutil.copy(best_path.parent.parent / "results.csv", OUT / f"training_{held}.csv") if (best_path.parent.parent / "results.csv").exists() else None
    shutil.rmtree(best_path.parent, ignore_errors=True)   # Keep the output small
    print(held, "predicted", len(list((preds / held).glob('*.png'))), "test patches", flush=True)

In [ ]:
shutil.make_archive(str(OUT / "preds"), "zip", preds)
print(os.path.getsize(OUT / "preds.zip") / 1e6, "MB")